# 06 - Observability + FastAPI: production deployment

**Aligns with**: S4 Sec. 7 | **Estimated time**: 30 minutes | **Estimated cost**: ~$0.05

## What this notebook covers

1. **LangSmith tracing**: every node, every LLM call, every retrieval is
   automatically observable
2. **FastAPI service**: wrap the pipeline as HTTP endpoints - the
   "production-ready" answer interviewers expect


## Part 1: LangSmith Tracing

The codebase has `@traceable` decorators throughout
(`src/captioners/vlm_captioner.py`, `src/retrievers/vector.py`,
`src/pipelines/query.py`). As long as `LANGSMITH_API_KEY` is in `.env`, traces
**flow to LangSmith automatically**. No code change required.


In [ ]:
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv()

print(f"LANGSMITH_API_KEY set? {bool(os.getenv('LANGSMITH_API_KEY'))}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '(default)')}")

In [ ]:
from src.core.config import configure_langsmith
configure_langsmith()

from src.pipelines.ingestion import IngestionPipeline
from src.pipelines.query import QueryPipeline
from src.chunkers import RecursiveChunker
from src.retrievers import VectorRetriever, BM25Retriever, HybridRetriever
from src.generators import RAGGenerator
from src.observability import CostTracker

cost = CostTracker()
pipeline = IngestionPipeline(cost_tracker=cost)
report = pipeline.ingest(str(ROOT / "data/uploads/wells_fargo.pdf"))
chunks = RecursiveChunker().chunk(report.document)

vec = VectorRetriever(
    persist_dir=str(ROOT / "tmp_chroma_06"), collection="lab_06",
    embeddings_cache_dir=ROOT / "cache/embeddings",
)
vec.reset(); vec.index(chunks)
bm = BM25Retriever(); bm.index(chunks)
hybrid = HybridRetriever(vec, bm)
qp = QueryPipeline(hybrid, RAGGenerator(cost_tracker=cost), cost_tracker=cost)

result = qp.query("What was Wells Fargo's Q4 2025 net income?")
print(f"Total cost: ${cost.total:.5f}")
print(f"Answer: {result['answer'][:200]}")
print("\nOpen https://smith.langchain.com to see the trace tree.")

## Trace tree you'll see in LangSmith

```
query_pipeline                                 850ms
  classify_query                                 5ms
  deep_retrieve                                420ms
    hybrid_retrieve                            415ms
      vector_search                            280ms
        OpenAIEmbeddings.embed_query           270ms
      bm25_search                                1ms
  rag_generate                                 420ms
    ChatOpenAI.invoke                          415ms ($0.000012)
```

Click into any node to see input, output, latency, tokens, cost.

**Interview scenario**: "A customer says the RAG answered wrong. How do you debug?"

> Take their query -> search LangSmith -> open the trace -> 99% of the time the
> issue is in retrieval. LangSmith turns debugging from guessing into seeing.


## Part 2: FastAPI Service

`src/api/server.py` exposes 3 endpoints:

```
GET  /health    -> status check
POST /ingest    -> upload + index a PDF
POST /query     -> ask a question, get answer + citations
```

Start the server:
```bash
uvicorn src.api.server:app --reload --port 8000
```


In [ ]:
import httpx
client = httpx.Client(base_url="http://127.0.0.1:8000", timeout=120)

try:
    r = client.get("/health")
    print(f"GET /health -> {r.status_code}")
    print(r.json())
except httpx.ConnectError:
    print("Server not running. Start: uvicorn src.api.server:app --port 8000")

In [ ]:
import json
r = client.post("/ingest", json={"path": str(ROOT / "data/uploads/wells_fargo.pdf")})
print(f"POST /ingest -> {r.status_code}")
print(json.dumps(r.json(), indent=2))

In [ ]:
r = client.post("/query", json={
    "question": "What was Wells Fargo's Q4 2025 net income?", "k": 5,
})
print(f"POST /query -> {r.status_code}\n")
data = r.json()
print(f"Answer: {data['answer']}\n")
print(f"Citations ({len(data['citations'])}):")
for c in data["citations"]:
    print(f"  - chunk_id={c['chunk_id'][:16]}, page={c['page_number']}")
    print(f"    {c['text_preview']}")

## Live curl - shows the service is real

```bash
curl http://127.0.0.1:8000/health

curl -X POST http://127.0.0.1:8000/ingest \
  -H "Content-Type: application/json" \
  -d '{"path": "/abs/path/to/wells_fargo.pdf"}'

curl -X POST http://127.0.0.1:8000/query \
  -H "Content-Type: application/json" \
  -d '{"question": "What was Q4 net income?"}'
```


## Whole-lab recap - elevator pitch for interviews

> "I built a production RAG service from scratch on three real earnings PDFs:
>
> - **Ingestion**: PyMuPDF + vision LLM captioning of tables/images +
>   3-layer content-addressed cache
> - **Chunking**: 3 strategies, with a `compare_strategies` tool to quantify
>   which fits a corpus
> - **Retrieval**: vector + BM25 + RRF + parent-child swap
> - **Generation**: LangGraph state machine with branching, automatic refusal
>   on empty retrieval
> - **Eval**: Ragas 4 metrics + custom claim-level hallucination detector,
>   30-question golden set across 5 categories
> - **Observability**: every component `@traceable`, full visibility in LangSmith
> - **Service**: FastAPI HTTP wrapper, `POST /ingest` and `POST /query`,
>   curl-able"

Next steps:
- Push the repo to GitHub
- Record a 5-minute demo video (LangSmith trace + curl-ing the service)
- On your resume, write 1-2 lines with numbers ("30-q golden set, faithfulness
  improved from 0.X to 0.Y after adding parent-child + hybrid")
